# Multi-Sector Batch Analysis

This notebook runs the primary cross-sector experiment for XLB, XLF, XLI, XLP, XLU, XLV and XLY. XLC and XLRE are excluded because their shorter histories would materially reduce temporal comparability with the long-history sector ETFs. The established XLK and XLE outputs are reused as protected reference outputs. The 0.05–0.20 range is the primary moderate-noise experiment; 0.50–1.00 are exploratory bounded-input stress tests.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the cloned repository.')

ROOT = find_project_root()

os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.venv' / '.matplotlib'))

sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sector_config import FEATURE_GROUPS, MODERATE_NOISE_INTENSITIES, NOISE_INTENSITIES, PREDICTORS, RANDOM_SEEDS, STRESS_TEST_INTENSITIES
from sector_robustness_pipeline import create_validation_output, hash_files, load_local_ohlcv_csv, run_sector_pipeline

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({'figure.dpi': 120})

DOWNLOAD_START = '2000-01-03'
DOWNLOAD_END = '2026-07-01'
PRIMARY_METRICS = ('balanced_accuracy', 'f1_score', 'roc_auc', 'average_precision')
ALL_METRICS = ('accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'average_precision')
SECTOR_METADATA = {
    'XLK': 'Technology',
    'XLE': 'Energy',
    'XLB': 'Materials',
    'XLF': 'Financials',
    'XLI': 'Industrials',
    'XLP': 'Consumer Staples',
    'XLU': 'Utilities',
    'XLV': 'Health Care',
    'XLY': 'Consumer Discretionary',
}
SECTOR_ORDER = ('XLK', 'XLE', 'XLB', 'XLF', 'XLI', 'XLP', 'XLU', 'XLV', 'XLY')
NEW_TICKERS = ('XLB', 'XLF', 'XLI', 'XLP', 'XLU', 'XLV', 'XLY')
MODERATE_LABEL = 'Primary moderate-noise experiment (0.05–0.20)'
STRESS_LABEL = 'Exploratory bounded-input stress tests (0.50–1.00)'

ROOT = ROOT.resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
for directory in (RAW_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RUN_START = time.perf_counter()


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def protected_inventory() -> list[Path]:
    paths: list[Path] = []
    paths.extend(p for p in sorted((ROOT / 'notebooks').glob('0[1-7]_*.ipynb')) if p.is_file())
    for base in (ROOT / 'data', ROOT / 'outputs'):
        for path in sorted(base.rglob('*')):
            if path.is_file() and path.name.startswith(('xlk', 'xle')):
                paths.append(path)
    paths.extend([
        ROOT / 'src' / 'sector_config.py',
        ROOT / 'src' / 'sector_robustness_pipeline.py',
        ROOT / 'tests' / 'test_sector_robustness_pipeline.py',
    ])
    return list(dict.fromkeys(paths))


def standardise_model_names(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out['model'] = out['model'].replace({'LogisticRegression': 'Logistic Regression'})
    return out


def read_existing_clean_metrics(ticker: str) -> pd.DataFrame:
    ticker = ticker.upper()
    if ticker == 'XLK':
        frame = pd.read_csv(TABLE_DIR / 'xlk_clean_model_comparison.csv')
        keep = ['model', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'average_precision']
        frame = frame.loc[:, keep]
        frame = standardise_model_names(frame)
        frame = frame.loc[frame['model'].isin(['Logistic Regression', 'Random Forest'])].reset_index(drop=True)
    elif ticker == 'XLE':
        frame = pd.read_csv(TABLE_DIR / 'xle_clean_baseline_metrics.csv')
        frame = standardise_model_names(frame)
    else:
        raise KeyError(ticker)
    frame.insert(0, 'ticker', ticker)
    frame.insert(1, 'sector', SECTOR_METADATA[ticker])
    return frame


def read_existing_noise_tables(ticker: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ticker = ticker.upper()
    summary = pd.read_csv(TABLE_DIR / f'{ticker.lower()}_noise_robustness_summary.csv')
    detailed = pd.read_csv(TABLE_DIR / f'{ticker.lower()}_noise_robustness_detailed.csv')
    clipping = pd.read_csv(TABLE_DIR / f'{ticker.lower()}_noise_clipping_diagnostics.csv')
    for frame in (summary, detailed, clipping):
        frame.insert(0, 'ticker', ticker)
        frame.insert(1, 'sector', SECTOR_METADATA[ticker])
    return summary, detailed, clipping


def read_existing_feature_dataset(ticker: str) -> pd.DataFrame:
    ticker = ticker.upper()
    frame = pd.read_csv(PROCESSED_DIR / f'{ticker.lower()}_feature_dataset.csv', parse_dates=['Date'])
    frame = frame.sort_values('Date', kind='stable').reset_index(drop=True)
    frame.insert(1 if 'ticker' in frame.columns else 0, 'ticker', ticker)
    frame.insert(2 if 'sector' not in frame.columns else 0, 'sector', SECTOR_METADATA[ticker])
    return frame

def established_threshold(ticker: str) -> float:
    ticker = ticker.upper()
    if ticker == 'XLK':
        table = pd.read_csv(TABLE_DIR / 'xlk_feasibility_summary.csv')
        return float(table.loc[table['metric'].eq('training_75th_percentile_threshold'), 'value'].iloc[0])
    if ticker == 'XLE':
        notebook = json.loads((ROOT / 'notebooks' / '06_xle_cross_sector_replication.ipynb').read_text())
        text = '\n'.join(''.join(output.get('text', [])) for cell in notebook['cells'] for output in cell.get('outputs', []))
        match = re.search(r'threshold ([0-9.]+) purged', text)
        if match is None:
            raise RuntimeError('XLE threshold could not be recovered from notebook 06 output.')
        return float(match.group(1))
    raise KeyError(ticker)


def extract_validation_summary_from_artifacts(ticker: str, raw: pd.DataFrame, feature: pd.DataFrame, clean: pd.DataFrame, summary: pd.DataFrame, detailed: pd.DataFrame, clipping: pd.DataFrame, source: str, threshold_reference: float | None = None) -> dict[str, object]:
    ticker = ticker.upper()
    feature = feature.sort_values('Date', kind='stable').reset_index(drop=True)
    raw_dates = pd.DatetimeIndex(raw.index)
    first_test_date = pd.Timestamp(feature.loc[feature['sample_period'].eq('test'), 'Date'].min())
    training = feature.loc[feature['sample_period'].eq('train')].copy()
    test = feature.loc[feature['sample_period'].eq('test')].copy()
    purge_dates = raw_dates[raw_dates < first_test_date][-5:]
    assert len(purge_dates) == 5
    assert purge_dates.max() < first_test_date
    assert pd.Timestamp('2021-03-10') == first_test_date
    assert not feature['Date'].isin(purge_dates).any()
    threshold = float(threshold_reference if threshold_reference is not None else training['future_rv_5d'].quantile(0.75))
    expected = (feature['future_rv_5d'] > threshold).astype(int).to_numpy()
    observed = feature['high_volatility'].astype(int).to_numpy()
    assert np.array_equal(expected, observed)
    for purge_date in purge_dates:
        position = raw_dates.get_loc(purge_date)
        future_window = raw_dates[position + 1: position + 6]
        assert len(future_window) == 5
        assert (future_window >= first_test_date).any()
    clean_filename = 'xlk_clean_model_comparison.csv' if ticker == 'XLK' else f'{ticker.lower()}_clean_baseline_metrics.csv'
    record = {
        'ticker': ticker,
        'sector': SECTOR_METADATA[ticker],
        'source': source,
        'status': 'PASS',
        'raw_sample_size': int(len(raw)),
        'modelling_sample_size': int(len(feature)),
        'training_sample_size': int(len(training)),
        'test_sample_size': int(len(test)),
        'first_date': raw_dates.min().strftime('%Y-%m-%d'),
        'final_date': raw_dates.max().strftime('%Y-%m-%d'),
        'first_test_date': first_test_date.strftime('%Y-%m-%d'),
        'purge_dates': json.dumps([d.strftime('%Y-%m-%d') for d in purge_dates]),
        'training_threshold': threshold,
        'training_class_prevalence': float(training['high_volatility'].mean()),
        'test_class_prevalence': float(test['high_volatility'].mean()),
        'model_fit_count': int(len(clean)),
        'clean_metric_row_count': int(len(clean)),
        'detailed_row_count': int(len(detailed)),
        'summary_row_count': int(len(summary)),
        'clipping_row_count': int(len(clipping)),
        'clipping_total': int(clipping['total_clipped_values'].sum()),
        'target_values_consistent': True,
        'purge_dates_count': int(len(purge_dates)),
        'raw_file_hash': sha256(RAW_DIR / f'{ticker.lower()}_daily_2000_2026.csv'),
        'feature_dataset_hash': sha256(PROCESSED_DIR / f'{ticker.lower()}_feature_dataset.csv'),
        'clean_metrics_hash': sha256(TABLE_DIR / clean_filename),
        'summary_hash': sha256(TABLE_DIR / f'{ticker.lower()}_noise_robustness_summary.csv'),
        'detailed_hash': sha256(TABLE_DIR / f'{ticker.lower()}_noise_robustness_detailed.csv'),
        'clipping_hash': sha256(TABLE_DIR / f'{ticker.lower()}_noise_clipping_diagnostics.csv'),
        'noise_range_moderate': MODERATE_LABEL,
        'noise_range_stress': STRESS_LABEL,
    }
    return record


def save_validation_report(record: dict[str, object], ticker: str) -> Path:
    path = TABLE_DIR / f'{ticker.lower()}_validation_report.csv'
    pd.DataFrame([record]).to_csv(path, index=False)
    return path


def paired_noise_metrics_table(record: dict[str, object], summary: pd.DataFrame) -> pd.DataFrame:
    frame = summary.copy()
    frame.insert(0, 'ticker', record['ticker'])
    frame.insert(1, 'sector', record['sector'])
    return frame


def aggregate_validation_frame(record: dict[str, object]) -> pd.DataFrame:
    return pd.DataFrame([record])


def summarise_clipping(clipping: pd.DataFrame, ticker: str) -> pd.DataFrame:
    ticker = ticker.upper()
    rows = []
    selected_counts = {group: len(features) for group, features in FEATURE_GROUPS.items()}
    for (feature_group, noise_intensity), group_frame in clipping.groupby(['feature_group', 'noise_intensity'], sort=False):
        total = group_frame['total_clipped_values'].astype(float)
        proportion = total / (selected_counts[feature_group] * len(group_frame))
        rows.append({
            'ticker': ticker,
            'sector': SECTOR_METADATA[ticker],
            'feature_group': feature_group,
            'noise_intensity': noise_intensity,
            'seed_count': int(len(group_frame)),
            'mean_total_clipped_values': float(total.mean()),
            'std_total_clipped_values': float(total.std(ddof=1)),
            'clipped_values_ci95_lower': float(total.mean() - 1.96 * total.std(ddof=1) / math.sqrt(len(group_frame))),
            'clipped_values_ci95_upper': float(total.mean() + 1.96 * total.std(ddof=1) / math.sqrt(len(group_frame))),
            'mean_clipped_proportion': float(proportion.mean()),
            'std_clipped_proportion': float(proportion.std(ddof=1)),
            'clipped_proportion_ci95_lower': float(proportion.mean() - 1.96 * proportion.std(ddof=1) / math.sqrt(len(group_frame))),
            'clipped_proportion_ci95_upper': float(proportion.mean() + 1.96 * proportion.std(ddof=1) / math.sqrt(len(group_frame))),
        })
    return pd.DataFrame(rows).sort_values(['ticker', 'feature_group', 'noise_intensity'], kind='stable').reset_index(drop=True)


def summarise_ranking(all_key: pd.DataFrame) -> pd.DataFrame:
    moderate = all_key.loc[
        all_key['feature_group'].eq('all_predictors')
        & all_key['noise_intensity'].isin(MODERATE_NOISE_INTENSITIES)
        & all_key['metric'].isin(PRIMARY_METRICS)
    ].copy()
    grouped = moderate.groupby(['ticker', 'sector', 'metric', 'model'], sort=False).agg(
        clean_performance=('clean_performance', 'first'),
        mean_performance_degradation=('mean_performance_degradation', 'mean'),
        std_performance_degradation=('mean_performance_degradation', 'std'),
    ).reset_index()
    rows = []
    for (ticker, sector, metric), block in grouped.groupby(['ticker', 'sector', 'metric'], sort=False):
        clean_winner_row = block.sort_values('clean_performance', ascending=False, kind='stable').iloc[0]
        robust_winner_row = block.sort_values('mean_performance_degradation', ascending=True, kind='stable').iloc[0]
        robust_loser_row = block.sort_values('mean_performance_degradation', ascending=True, kind='stable').iloc[-1]
        rows.append({
            'ticker': ticker,
            'sector': sector,
            'metric': metric,
            'clean_winner': clean_winner_row['model'],
            'clean_winner_performance': float(clean_winner_row['clean_performance']),
            'clean_loser_performance': float(block.loc[block['model'] != clean_winner_row['model'], 'clean_performance'].iloc[0]),
            'clean_margin': float(abs(block['clean_performance'].max() - block['clean_performance'].min())),
            'robustness_winner': robust_winner_row['model'],
            'robustness_winner_mean_degradation': float(robust_winner_row['mean_performance_degradation']),
            'robustness_loser_mean_degradation': float(robust_loser_row['mean_performance_degradation']),
            'robustness_margin': float(robust_loser_row['mean_performance_degradation'] - robust_winner_row['mean_performance_degradation']),
            'winner_match': bool(clean_winner_row['model'] == robust_winner_row['model']),
        })
    return pd.DataFrame(rows).sort_values(['ticker', 'metric'], kind='stable').reset_index(drop=True)


def standardise_existing_summary(ticker: str, summary: pd.DataFrame) -> pd.DataFrame:
    frame = summary.copy()
    frame.insert(0, 'ticker', ticker)
    frame.insert(1, 'sector', SECTOR_METADATA[ticker])
    return frame


def standardise_existing_detailed(ticker: str, detailed: pd.DataFrame) -> pd.DataFrame:
    frame = detailed.copy()
    frame.insert(0, 'ticker', ticker)
    frame.insert(1, 'sector', SECTOR_METADATA[ticker])
    return frame


def standardise_existing_clipping(ticker: str, clipping: pd.DataFrame) -> pd.DataFrame:
    frame = clipping.copy()
    frame.insert(0, 'ticker', ticker)
    frame.insert(1, 'sector', SECTOR_METADATA[ticker])
    return frame

In [ ]:
protected_paths = protected_inventory()
protected_before = {str(path.relative_to(ROOT)): sha256(path) for path in protected_paths}
created_files: list[str] = []
sector_records: dict[str, dict[str, object]] = {}
sector_validation_rows: list[pd.DataFrame] = []
sector_clean_frames: list[pd.DataFrame] = []
sector_summary_frames: list[pd.DataFrame] = []
sector_detailed_frames: list[pd.DataFrame] = []
sector_clipping_frames: list[pd.DataFrame] = []
failed_sectors: list[dict[str, object]] = []

for ticker in NEW_TICKERS:
    ticker_lower = ticker.lower()
    raw_path = RAW_DIR / f'{ticker_lower}_daily_2000_2026.csv'
    feature_path = PROCESSED_DIR / f'{ticker_lower}_feature_dataset.csv'
    clean_path = TABLE_DIR / f'{ticker_lower}_clean_baseline_metrics.csv'
    detailed_path = TABLE_DIR / f'{ticker_lower}_noise_robustness_detailed.csv'
    summary_path = TABLE_DIR / f'{ticker_lower}_noise_robustness_summary.csv'
    clipping_path = TABLE_DIR / f'{ticker_lower}_noise_clipping_diagnostics.csv'
    validation_path = TABLE_DIR / f'{ticker_lower}_validation_report.csv'
    try:
        print(f'Running {ticker}...')
        if all(path.exists() for path in [raw_path, feature_path, clean_path, detailed_path, summary_path, clipping_path, validation_path]):
            raw = load_local_ohlcv_csv(raw_path, ticker)
            feature = pd.read_csv(feature_path, parse_dates=['Date']).sort_values('Date', kind='stable').reset_index(drop=True)
            clean = pd.read_csv(clean_path)
            summary = pd.read_csv(summary_path)
            detailed = pd.read_csv(detailed_path)
            clipping = pd.read_csv(clipping_path)
            validation = pd.read_csv(validation_path).iloc[0].to_dict()
            validation['sector'] = SECTOR_METADATA[ticker]
            validation['source'] = 'cache'
            sector_records[ticker] = {
                'raw': raw,
                'feature': feature,
                'clean': clean,
                'summary': summary,
                'detailed': detailed,
                'clipping': clipping,
                'validation': validation,
                'raw_path': raw_path,
                'feature_path': feature_path,
                'clean_path': clean_path,
                'detailed_path': detailed_path,
                'summary_path': summary_path,
                'clipping_path': clipping_path,
                'validation_path': validation_path,
            }
            sector_clean_frames.append(clean.assign(ticker=ticker, sector=SECTOR_METADATA[ticker]))
            sector_summary_frames.append(paired_noise_metrics_table(validation, summary))
            sector_detailed_frames.append(standardise_existing_detailed(ticker, detailed))
            sector_clipping_frames.append(standardise_existing_clipping(ticker, clipping))
            sector_validation_rows.append(pd.DataFrame([validation]))
            created_files.extend([str(p.relative_to(ROOT)) for p in [raw_path, feature_path, clean_path, detailed_path, summary_path, clipping_path, validation_path]])
            print(f"  reused cached outputs; raw rows {validation['raw_sample_size']}; modelling rows {validation['modelling_sample_size']}; threshold {validation['training_threshold']:.15f}")
            continue

        downloaded = yf.download(ticker, start=DOWNLOAD_START, end=DOWNLOAD_END, auto_adjust=False, progress=False)
        if downloaded.empty:
            raise RuntimeError(f'{ticker}: yfinance returned no data.')
        downloaded.to_csv(raw_path)
        raw = load_local_ohlcv_csv(raw_path, ticker)
        dataset, results = run_sector_pipeline(raw, ticker)

        dataset.modelling.to_csv(feature_path, index=False)
        results.clean_metrics.to_csv(clean_path, index=False)
        results.detailed.to_csv(detailed_path, index=False)
        results.summary.to_csv(summary_path, index=False)
        results.clipping.to_csv(clipping_path, index=False)

        validation = create_validation_output(
            dataset=dataset,
            results=results,
            protected_hashes=protected_before,
            output_hashes=hash_files([feature_path, clean_path, detailed_path, summary_path, clipping_path], ticker),
        )
        validation['raw_file_hash'] = sha256(raw_path)
        validation['sector'] = SECTOR_METADATA[ticker]
        validation['source'] = 'new'
        validation_path = save_validation_report(validation, ticker)
        validation['validation_report_hash'] = sha256(validation_path)

        sector_records[ticker] = {
            'raw': raw,
            'dataset': dataset,
            'results': results,
            'validation': validation,
            'raw_path': raw_path,
            'feature_path': feature_path,
            'clean_path': clean_path,
            'detailed_path': detailed_path,
            'summary_path': summary_path,
            'clipping_path': clipping_path,
            'validation_path': validation_path,
        }

        sector_clean_frames.append(results.clean_metrics.assign(ticker=ticker, sector=SECTOR_METADATA[ticker]))
        sector_summary_frames.append(paired_noise_metrics_table(validation, results.summary))
        sector_detailed_frames.append(standardise_existing_detailed(ticker, results.detailed))
        sector_clipping_frames.append(standardise_existing_clipping(ticker, results.clipping))
        sector_validation_rows.append(aggregate_validation_frame(validation))
        created_files.extend([str(p.relative_to(ROOT)) for p in [raw_path, feature_path, clean_path, detailed_path, summary_path, clipping_path, validation_path]])
        print(f'  downloaded {len(raw)} raw rows; modelling rows {len(dataset.modelling)}; threshold {dataset.training_threshold:.15f}')
        print(f'  clean rows {len(results.clean_metrics)}; detailed rows {len(results.detailed)}; summary rows {len(results.summary)}; clipping rows {len(results.clipping)}')
    except Exception as exc:
        failed_sectors.append({'ticker': ticker, 'sector': SECTOR_METADATA[ticker], 'status': 'FAIL', 'error': str(exc)})
        print(f'  FAILED: {exc}')

# Existing XLK and XLE artefacts are reused as protected references.
for ticker in ('XLK', 'XLE'):
    raw = load_local_ohlcv_csv(RAW_DIR / f'{ticker.lower()}_daily_2000_2026.csv', ticker)
    feature = read_existing_feature_dataset(ticker)
    clean = read_existing_clean_metrics(ticker)
    summary, detailed, clipping = read_existing_noise_tables(ticker)
    validation = extract_validation_summary_from_artifacts(
        ticker, raw, feature, clean, summary, detailed, clipping, source='established',
        threshold_reference=established_threshold(ticker)
    )
    sector_records[ticker] = {
        'raw': raw,
        'feature': feature,
        'clean': clean,
        'summary': summary,
        'detailed': detailed,
        'clipping': clipping,
        'validation': validation,
    }
    sector_clean_frames.append(clean)
    sector_summary_frames.append(summary)
    sector_detailed_frames.append(detailed)
    sector_clipping_frames.append(clipping)
    sector_validation_rows.append(aggregate_validation_frame(validation))

print(f'Completed sectors: {sorted(sector_records)}')
if failed_sectors:
    print('Failures recorded:')
    for failure in failed_sectors:
        print(json.dumps(failure, indent=2))
else:
    print('No sector failures recorded.')

In [ ]:
all_clean_metrics = pd.concat(sector_clean_frames, ignore_index=True)
all_clean_metrics = all_clean_metrics.sort_values(['ticker', 'model'], kind='stable').reset_index(drop=True)
all_clean_metrics.to_csv(TABLE_DIR / 'all_sector_clean_metrics.csv', index=False)
created_files.append('outputs/tables/all_sector_clean_metrics.csv')

all_noise_summary = pd.concat(sector_summary_frames, ignore_index=True)
all_noise_summary = all_noise_summary.sort_values(['ticker', 'model', 'feature_group', 'noise_intensity', 'metric'], kind='stable').reset_index(drop=True)
all_noise_summary.to_csv(TABLE_DIR / 'all_sector_noise_summary.csv', index=False)
created_files.append('outputs/tables/all_sector_noise_summary.csv')

all_key_degradation = all_noise_summary.loc[all_noise_summary['metric'].isin(PRIMARY_METRICS)].copy()
all_key_degradation = all_key_degradation.sort_values(['ticker', 'model', 'feature_group', 'noise_intensity', 'metric'], kind='stable').reset_index(drop=True)
all_key_degradation.to_csv(TABLE_DIR / 'all_sector_key_degradation.csv', index=False)
created_files.append('outputs/tables/all_sector_key_degradation.csv')

all_clipping_detail = pd.concat(sector_clipping_frames, ignore_index=True)
all_clipping_detail = all_clipping_detail.sort_values(['ticker', 'feature_group', 'noise_intensity'], kind='stable').reset_index(drop=True)

test_size_lookup = {
    ticker: int(record['validation']['test_sample_size'])
    for ticker, record in sector_records.items()
}

clipping_summary_rows = []
for (ticker, feature_group, noise_intensity), group_frame in all_clipping_detail.groupby(['ticker', 'feature_group', 'noise_intensity'], sort=False):
    selected_count = len(FEATURE_GROUPS[feature_group])
    denominator = selected_count * test_size_lookup[ticker]
    proportion = group_frame['total_clipped_values'].astype(float) / denominator
    total = group_frame['total_clipped_values'].astype(float)
    total_sd = total.std(ddof=1)
    prop_sd = proportion.std(ddof=1)
    clipping_summary_rows.append({
        'ticker': ticker,
        'sector': SECTOR_METADATA[ticker],
        'feature_group': feature_group,
        'noise_intensity': noise_intensity,
        'seed_count': int(len(group_frame)),
        'mean_total_clipped_values': float(total.mean()),
        'std_total_clipped_values': float(total_sd),
        'clipped_values_ci95_lower': float(total.mean() - 1.96 * total_sd / math.sqrt(len(group_frame))),
        'clipped_values_ci95_upper': float(total.mean() + 1.96 * total_sd / math.sqrt(len(group_frame))),
        'mean_clipped_proportion': float(proportion.mean()),
        'std_clipped_proportion': float(prop_sd),
        'clipped_proportion_ci95_lower': float(proportion.mean() - 1.96 * prop_sd / math.sqrt(len(group_frame))),
        'clipped_proportion_ci95_upper': float(proportion.mean() + 1.96 * prop_sd / math.sqrt(len(group_frame))),
    })
all_clipping_summary = pd.DataFrame(clipping_summary_rows).sort_values(['ticker', 'feature_group', 'noise_intensity'], kind='stable').reset_index(drop=True)
all_clipping_summary.to_csv(TABLE_DIR / 'all_sector_clipping_summary.csv', index=False)
created_files.append('outputs/tables/all_sector_clipping_summary.csv')

all_validation_summary = pd.concat(sector_validation_rows, ignore_index=True)
all_validation_summary = all_validation_summary.sort_values(['ticker'], kind='stable').reset_index(drop=True)
all_validation_summary.to_csv(TABLE_DIR / 'all_sector_validation_summary.csv', index=False)
created_files.append('outputs/tables/all_sector_validation_summary.csv')

all_ranking = summarise_ranking(all_key_degradation)
all_ranking.to_csv(TABLE_DIR / 'all_sector_model_robustness_ranking.csv', index=False)
created_files.append('outputs/tables/all_sector_model_robustness_ranking.csv')

created_files = list(dict.fromkeys(created_files))

sector_order = list(SECTOR_ORDER)
metric_titles = {
    'balanced_accuracy': 'Balanced accuracy',
    'f1_score': 'F1-score',
    'roc_auc': 'ROC-AUC',
    'average_precision': 'Average precision',
}

# Figure 1: clean model performance.
clean_plot = all_clean_metrics.loc[all_clean_metrics['model'].isin(['Logistic Regression', 'Random Forest'])].copy()
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharey=False)
for axis, metric in zip(axes.flat, PRIMARY_METRICS, strict=True):
    metric_frame = clean_plot[['ticker', 'sector', 'model', metric]].copy()
    metric_frame['ticker'] = pd.Categorical(metric_frame['ticker'], categories=sector_order, ordered=True)
    metric_frame = metric_frame.sort_values(['ticker', 'model'], kind='stable')
    sns.barplot(data=metric_frame, x='ticker', y=metric, hue='model', ax=axis, palette=['#355C7D', '#C06C84'])
    axis.set_title(metric_titles[metric])
    axis.set_xlabel('Sector ETF')
    axis.set_ylabel('Performance')
    axis.set_ylim(max(0, metric_frame[metric].min() - 0.05), min(1, metric_frame[metric].max() + 0.05))
    axis.legend(title='Model')
    axis.tick_params(axis='x', rotation=45)
fig.suptitle('Clean test performance by sector ETF and model')
fig.tight_layout()
clean_fig_path = FIGURE_DIR / 'all_sector_clean_model_performance.png'
fig.savefig(clean_fig_path, dpi=300, bbox_inches='tight')
plt.close(fig)
created_files.append('outputs/figures/all_sector_clean_model_performance.png')

# Figure 2: all-predictor degradation with sectors, models and intensities.
all_predictor_key = all_key_degradation.loc[all_key_degradation['feature_group'].eq('all_predictors')].copy()
all_predictor_key['ticker'] = pd.Categorical(all_predictor_key['ticker'], categories=sector_order, ordered=True)
fig, axes = plt.subplots(4, 2, figsize=(20, 22), sharex=True)
for row_index, metric in enumerate(PRIMARY_METRICS):
    metric_frame = all_predictor_key.loc[all_predictor_key['metric'].eq(metric)].copy()
    value_min = float(metric_frame['degradation_ci95_lower'].min())
    value_max = float(metric_frame['degradation_ci95_upper'].max())
    margin = max((value_max - value_min) * 0.08, 0.002)
    vmin = value_min - margin
    vmax = value_max + margin
    for col_index, model in enumerate(['Logistic Regression', 'Random Forest']):
        axis = axes[row_index, col_index]
        panel = metric_frame.loc[metric_frame['model'].eq(model)].copy()
        heat = panel.pivot(index='ticker', columns='noise_intensity', values='mean_performance_degradation').reindex(index=sector_order, columns=list(NOISE_INTENSITIES))
        sns.heatmap(heat, ax=axis, cmap='vlag', center=0, vmin=vmin, vmax=vmax, annot=True, fmt='.3f', cbar=col_index == 1, cbar_kws={'label': 'Mean degradation' if col_index == 1 else ''})
        axis.set_title(f'{metric_titles[metric]} - {model}')
        axis.set_xlabel('Noise intensity')
        axis.set_ylabel('Sector ETF' if col_index == 0 else '')
        axis.tick_params(axis='x', rotation=0)
        axis.tick_params(axis='y', rotation=0)
fig.suptitle('All-predictor mean degradation by sector ETF, model and noise intensity')
fig.tight_layout()
all_pred_fig_path = FIGURE_DIR / 'all_sector_all_predictor_degradation.png'
fig.savefig(all_pred_fig_path, dpi=300, bbox_inches='tight')
plt.close(fig)
created_files.append('outputs/figures/all_sector_all_predictor_degradation.png')

# Figure 3: feature-group degradation heatmap averaged over the moderate-noise range.
moderate_feature = all_key_degradation.loc[all_key_degradation['noise_intensity'].isin(MODERATE_NOISE_INTENSITIES) & all_key_degradation['metric'].isin(PRIMARY_METRICS)].copy()
fig, axes = plt.subplots(4, 2, figsize=(20, 22), sharex=True)
for row_index, metric in enumerate(PRIMARY_METRICS):
    metric_frame = moderate_feature.loc[moderate_feature['metric'].eq(metric)].copy()
    value_min = float(metric_frame['degradation_ci95_lower'].min())
    value_max = float(metric_frame['degradation_ci95_upper'].max())
    margin = max((value_max - value_min) * 0.08, 0.002)
    vmin = value_min - margin
    vmax = value_max + margin
    for col_index, model in enumerate(['Logistic Regression', 'Random Forest']):
        axis = axes[row_index, col_index]
        panel = metric_frame.loc[metric_frame['model'].eq(model)].copy()
        panel = panel.groupby(['ticker', 'sector', 'feature_group'], sort=False)['mean_performance_degradation'].mean().reset_index()
        heat = panel.pivot(index='ticker', columns='feature_group', values='mean_performance_degradation').reindex(index=sector_order, columns=list(FEATURE_GROUPS))
        sns.heatmap(heat, ax=axis, cmap='vlag', center=0, vmin=vmin, vmax=vmax, annot=True, fmt='.3f', cbar=col_index == 1, cbar_kws={'label': 'Mean degradation' if col_index == 1 else ''})
        axis.set_title(f'{metric_titles[metric]} - {model}')
        axis.set_xlabel('Feature group')
        axis.set_ylabel('Sector ETF' if col_index == 0 else '')
        axis.tick_params(axis='x', rotation=20)
        axis.tick_params(axis='y', rotation=0)
fig.suptitle('Moderate-noise mean degradation by sector ETF, model and feature group')
fig.tight_layout()
feature_heatmap_path = FIGURE_DIR / 'all_sector_feature_group_degradation_heatmap.png'
fig.savefig(feature_heatmap_path, dpi=300, bbox_inches='tight')
plt.close(fig)
created_files.append('outputs/figures/all_sector_feature_group_degradation_heatmap.png')

# Figure 4: clean vs robustness winner comparison.
comparison = all_ranking.copy()
comparison['clean_code'] = comparison['clean_winner'].map({'Logistic Regression': 0, 'Random Forest': 1})
comparison['robustness_code'] = comparison['robustness_winner'].map({'Logistic Regression': 0, 'Random Forest': 1})
fig, axes = plt.subplots(1, 2, figsize=(18, 10), sharey=True)
for axis, column, title in zip(axes, ['clean_code', 'robustness_code'], ['Clean-performance winner', 'Robustness winner (moderate all-predictor noise)'], strict=True):
    pivot = comparison.pivot(index='ticker', columns='metric', values=column).reindex(index=sector_order, columns=PRIMARY_METRICS)
    annotation = comparison.pivot(index='ticker', columns='metric', values='clean_winner').reindex(index=sector_order, columns=PRIMARY_METRICS).replace({'Logistic Regression': 'LR', 'Random Forest': 'RF'})
    sns.heatmap(pivot, ax=axis, cmap=sns.color_palette(['#355C7D', '#C06C84'], as_cmap=True), vmin=0, vmax=1, annot=annotation, fmt='', cbar=False)
    axis.set_title(title)
    axis.set_xlabel('Metric')
    axis.set_ylabel('Sector ETF')
    axis.tick_params(axis='x', rotation=20)
    axis.tick_params(axis='y', rotation=0)
handles = [plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='#355C7D', markersize=12, label='Logistic Regression'), plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='#C06C84', markersize=12, label='Random Forest')]
fig.legend(handles=handles, loc='lower center', ncol=2)
fig.suptitle('Clean and robustness winners by sector ETF and metric')
fig.tight_layout(rect=(0, 0.05, 1, 1))
comparison_fig_path = FIGURE_DIR / 'all_sector_model_robustness_comparison.png'
fig.savefig(comparison_fig_path, dpi=300, bbox_inches='tight')
plt.close(fig)
created_files.append('outputs/figures/all_sector_model_robustness_comparison.png')

# Figure 5: clipping proportions by feature group.
clipping_plot = all_clipping_summary.copy()
fig, axes = plt.subplots(3, 2, figsize=(20, 18), sharex=True)
axes = axes.flatten()
all_vmin = float(clipping_plot['clipped_proportion_ci95_lower'].min())
all_vmax = float(clipping_plot['clipped_proportion_ci95_upper'].max())
margin = max((all_vmax - all_vmin) * 0.08, 0.002)
for axis, feature_group in zip(axes, list(FEATURE_GROUPS), strict=False):
    panel = clipping_plot.loc[clipping_plot['feature_group'].eq(feature_group)].copy()
    heat = panel.pivot(index='ticker', columns='noise_intensity', values='mean_clipped_proportion').reindex(index=sector_order, columns=list(NOISE_INTENSITIES))
    sns.heatmap(heat, ax=axis, cmap='mako', vmin=all_vmin - margin, vmax=all_vmax + margin, annot=True, fmt='.3f', cbar=axis is axes[-1], cbar_kws={'label': 'Mean clipped proportion' if axis is axes[-1] else ''})
    axis.set_title(feature_group.replace('_', ' ').title())
    axis.set_xlabel('Noise intensity')
    axis.set_ylabel('Sector ETF')
    axis.tick_params(axis='x', rotation=0)
    axis.tick_params(axis='y', rotation=0)
for axis in axes[len(FEATURE_GROUPS):]:
    axis.remove()
fig.suptitle('Logical-bound clipping proportion by sector ETF and feature group')
fig.tight_layout()
clipping_fig_path = FIGURE_DIR / 'all_sector_clipping_proportions.png'
fig.savefig(clipping_fig_path, dpi=300, bbox_inches='tight')
plt.close(fig)
created_files.append('outputs/figures/all_sector_clipping_proportions.png')

created_files = list(dict.fromkeys(created_files))

print(f'Aggregate clean rows: {len(all_clean_metrics)}')
print(f'Aggregate noise-summary rows: {len(all_noise_summary)}')
print(f'Aggregate key-degradation rows: {len(all_key_degradation)}')
print(f'Aggregate validation rows: {len(all_validation_summary)}')
print(f'Aggregate ranking rows: {len(all_ranking)}')
print(f'Aggregate clipping-summary rows: {len(all_clipping_summary)}')

In [ ]:
protected_after = {str(path.relative_to(ROOT)): sha256(path) for path in protected_paths}
assert protected_after == protected_before, 'A protected notebook or analytical artefact changed during the batch run.'

# Final numerical checks for the completed sectors.
if failed_sectors:
    print('One or more sectors failed and were excluded from the aggregate outputs.')
    print(json.dumps(failed_sectors, indent=2))
else:
    print('All seven new sector ETFs completed successfully.')

print()
print('Download status for all seven ETFs:')
for ticker in NEW_TICKERS:
    record = sector_records.get(ticker)
    if record is None:
        print(f'- {ticker}: FAILED')
    else:
        validation = record['validation']
        print(f"- {ticker}: raw rows {validation['raw_sample_size']}; training {validation['training_sample_size']}; test {validation['test_sample_size']}; purge dates {validation['purge_dates']}; threshold {validation['training_threshold']:.15f}")
        print(f"  clean metrics rows {len(record['clean'])}; detailed rows {len(record['detailed'])}; summary rows {len(record['summary'])}; clipping rows {len(record['clipping'])}; clipping total {validation.get('clipping_total', int(record['clipping']['total_clipped_values'].sum()))}")

print()
print('Existing XLK and XLE validation status:')
for ticker in ('XLK', 'XLE'):
    record = sector_records[ticker]['validation']
    print(f"- {ticker}: PASS; threshold {record['training_threshold']:.15f}; training {record['training_sample_size']}; test {record['test_sample_size']}; purge dates {record['purge_dates']}")

print()
print('Aggregate files created:')
for rel in [
    'outputs/tables/all_sector_clean_metrics.csv',
    'outputs/tables/all_sector_noise_summary.csv',
    'outputs/tables/all_sector_key_degradation.csv',
    'outputs/tables/all_sector_validation_summary.csv',
    'outputs/tables/all_sector_model_robustness_ranking.csv',
    'outputs/tables/all_sector_clipping_summary.csv',
    'outputs/figures/all_sector_clean_model_performance.png',
    'outputs/figures/all_sector_all_predictor_degradation.png',
    'outputs/figures/all_sector_feature_group_degradation_heatmap.png',
    'outputs/figures/all_sector_model_robustness_comparison.png',
    'outputs/figures/all_sector_clipping_proportions.png',
]:
    print(f'- {rel}')

print()
print('Protected files remained unchanged.')
print(f'Total execution time: {time.perf_counter() - RUN_START:.1f} seconds')